# Smart Grid Power Demand Forecasting
## Notebook 01: Data Ingestion, Cleaning & Exploratory Data Analysis (EDA)

### Project Objective
To accurately forecast hourly electricity demand (**Demand (MW)**) for the next **24 hours** using Indian electricity grid hourly data.

### Core Engineering Rules Followed
1. **Raw Data Immutability:** Raw Excel files remain untouched in `dataset/raw/`.
2. **Strict Chronological Splitting:** Never use random shuffle split for time series.
3. **Anti-Leakage Guarantees:** No future data statistics used in preprocessing or transformations.
4. **No Weather Data Assumed:** Exclusively utilizes the grid telemetry columns provided.
5. **Zero Data Invention:** Missing values are not invented; unusual readings are audited and physically documented without silent dropping.

### 1. Environment & Modular Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path for modular src imports
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_PATH,
    PROCESSED_DATA_CSV,
    COL_TIMESTAMP,
    COL_DEMAND,
    COL_WIND,
    COL_SOLAR,
    COL_TOTAL_GEN,
    TRAIN_END_DATE,
    VAL_END_DATE,
)
from src.data_loader import inspect_raw_structure, load_raw_dataset
from src.cleaning import clean_grid_data
from src.eda import compute_statistical_summary

sns.set_theme(style="whitegrid", font_scale=1.05)
print("Project environment initialized successfully.")

### 2. Raw Data Inspection & Structural Audit
Let us inspect the raw Excel workbook structure and verify footer metadata separation.

In [ ]:
raw_info = inspect_raw_structure(RAW_DATA_PATH)
for key, val in raw_info.items():
    if key != 'footer_sample':
        print(f"{key}: {val}")

print("\n--- Footer Rows Detected (Isolated from time-series observations) ---")
for row in raw_info['footer_sample']:
    print(row)

### 3. Data Cleaning and Continuity Verification
We now clean the hourly observations:
- Parse timestamp format `DD-MM-YYYY HH:MM:SS`
- Verify exact chronological continuity (12,984 hours from 2024-01-01 to 2025-06-24)
- Audit solar readings (negative values explained by nighttime inverter standby load)

In [ ]:
df = clean_grid_data(raw_path=RAW_DATA_PATH, save_processed=True)
print("Cleaned DataFrame Shape:", df.shape)
df.head()

### 4. Descriptive Statistics & Split Verification

In [ ]:
summary = compute_statistical_summary(df)
splits_df = pd.DataFrame(summary['splits']).T
print("Chronological Splitting Scheme:")
splits_df

### 5. Exploratory Data Analysis Visualizations
#### 5.1 Chronological Time-Series & Splits (No Leakage)

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(df[COL_TIMESTAMP], df[COL_DEMAND] / 1000.0, lw=0.8, color='#1f77b4', label='Demand (GW)')
plt.axvline(pd.to_datetime(TRAIN_END_DATE), color='red', linestyle='--', lw=2, label='Train Split End')
plt.axvline(pd.to_datetime(VAL_END_DATE), color='green', linestyle='--', lw=2, label='Val Split End')
plt.title('Indian Electricity Grid Demand Time Series with Strict Chronological Splits')
plt.ylabel('Demand (GW)')
plt.xlabel('Date')
plt.legend()
plt.show()

#### 5.2 24-Hour Diurnal Demand Profile
Shows the daily peak and off-peak rhythms critical for the 24-hour ahead forecasting horizon.

In [ ]:
df['Hour'] = df[COL_TIMESTAMP].dt.hour
hourly_demand = df.groupby('Hour')[COL_DEMAND].agg(['mean', 'median', lambda x: x.quantile(0.10), lambda x: x.quantile(0.90)])

plt.figure(figsize=(12, 5))
plt.plot(hourly_demand.index, hourly_demand['mean'] / 1000.0, marker='o', color='#d62728', lw=2.5, label='Mean Demand (GW)')
plt.fill_between(hourly_demand.index, hourly_demand['<lambda_0>'] / 1000.0, hourly_demand['<lambda_1>'] / 1000.0, color='#d62728', alpha=0.15, label='10th–90th Percentile')
plt.title('24-Hour Diurnal Demand Curve (Average Daily Profile)')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('Demand (GW)')
plt.xticks(range(0, 24))
plt.legend()
plt.show()

#### 5.3 Correlation Matrix & Generation Composition

In [ ]:
corr_cols = [COL_DEMAND, COL_TOTAL_GEN, COL_WIND, 'Solar_Calibrated (MW)', 'Net_Demand (MW)']
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.3f', cmap='vlag', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()